In [1]:
from dotenv import load_dotenv
from pathlib import Path
from openai import OpenAI

load_dotenv()

True

In [61]:
import os
open_ai_api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=open_ai_api_key)

In [62]:
def call_openai(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.choices[0].message.content

In [64]:
pr ={
    "number": 123,
    "comments":[],
    "status":"PENDING"
}

In [65]:
## tools
def read_file(file_path):
    with open(file_path, "r") as f:
        return f.read()

def post_review_comment(filename, comment):
    # Placeholder for posting a review comment
    pr["comments"].append({"filename": filename, "comment": comment})


def set_pr_status(status):
    if status not in ["PASS", "BLOCKED"]:
        raise ValueError("Invalid status. Must be one of: PASS, BLOCKED")
    pr["status"] = status
    return f"PR status set to {status}"
    

In [66]:
read_file("./data/team_standards.md")

'# Team Coding Standards\n\n1. All DB queries must use parameterized queries (SQLAlchemy)\n2. Never log sensitive data (card numbers, passwords, tokens)\n3. All user inputs must be validated before DB queries\n4. Auth tokens must use JWT with expiry\n5. All payment operations must be idempotent\n6. All order ids should be masked in the return statement'

In [67]:
tools =[
   { 
       "type":"function",
        "function":{
            "name":"read_file",
            "description":"Reads the content of a file given its path.",
            "parameters":{
                "type":"object",
                "properties":{
                    "file_path":{
                        "type":"string",
                        "description":"The path to the file to be read."
                    }
                },
                "required":["file_path"]
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"post_review_comment",
            "description":"Posts a review comment on a specific file in the PR.",
            "parameters":{
                "type":"object",
                "properties":{
                    "filename":{
                        "type":"string",
                        "description":"The name of the file to comment on."
                    },
                    "comment":{
                        "type":"string",
                        "description":"The content of the review comment."
                    }
                },
                "required":["filename", "comment"]
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"set_pr_status",
            "description":"Sets the status of the PR to either PASS or BLOCKED.",
            "parameters":{
                "type":"object",
                "properties":{
                    "status":{
                        "type":"string",
                        "enum":["PASS", "BLOCKED"],
                        "description":"The new status for the PR."
                    }
                },
                "required":["status"]
            }
        }
    }
]

In [68]:
demo_messages=[
    {"role":"user", "content":"Please read my auth.py before reviewing it which is at location ./data/auth.py."}
]

client= OpenAI(api_key=open_ai_api_key)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=demo_messages,
    tools=tools,
)
response.choices[0].message.content

In [69]:
print(response)

ChatCompletion(id='chatcmpl-Ef3kfIY1ZP5D0QUSUc6HbjEWVdhIb', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Xy6Z3g9EeMgGxansDJHJiW6N', function=Function(arguments='{"file_path":"./data/auth.py"}', name='read_file'), type='function')]))], created=1793454577, model='gpt-4o-mini-2024-07-18', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_7e62990de7', usage=CompletionUsage(completion_tokens=18, prompt_tokens=165, total_tokens=183, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))


In [70]:
first_message = response.choices[0].message
demo_messages.append(first_message)

In [77]:
print(first_message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Xy6Z3g9EeMgGxansDJHJiW6N', function=Function(arguments='{"file_path":"./data/auth.py"}', name='read_file'), type='function')])


In [71]:
functionDetails= response.choices[0].message.tool_calls[0].function
call=response.choices[0].message.tool_calls[0]
print(functionDetails)

Function(arguments='{"file_path":"./data/auth.py"}', name='read_file')


In [72]:
import json

function_arguments = json.loads(functionDetails.arguments)
print(function_arguments)

{'file_path': './data/auth.py'}


In [73]:
resultOfreadFile =read_file(function_arguments["file_path"])

In [74]:
demo_messages.append({"role":"tool",  "tool_call_id": call.id, "content":resultOfreadFile})

In [75]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=demo_messages,
    tools=tools,
)

In [76]:
print(response.choices[0].message.content)

The `auth.py` file contains two functions:

1. **login(username, password)**: This function checks if the provided username and password match the hardcoded credentials (`admin` and `admin123`). It returns `True` if they match, and `False` otherwise.

2. **generate_token(user_id)**: This function generates a token for a given user ID by appending the current time (in seconds since the epoch) to the user ID, separated by an underscore.

### Review Comments:

1. **Security Concerns**:
   - Hardcoding credentials in the `login` function is a security risk. This approach should be avoided, and a secure authentication mechanism should be employed.
   - The password should ideally be hashed and compared against a stored hash rather than using plain text.

2. **Token Generation**:
   - The token generated in `generate_token` includes a timestamp, which may not be unique over time for the same user ID. Consider implementing a more robust token generation method (e.g., using JWTs or UUIDs).

3.

# FIRST Code Review Agent

In [88]:
## tools
def read_file(file_path):
    with open(file_path, "r") as f:
        return f.read()

def post_review_comment(filename, comment):
    # Placeholder for posting a review comment
    pr["comments"].append({"filename": filename, "comment": comment})


def set_pr_status(status):
    if status not in ["PASS", "BLOCKED"]:
        raise ValueError("Invalid status. Must be one of: PASS, BLOCKED")
    pr["status"] = status
    return f"PR status set to {status}"

def get_files_in_pr():
    # Placeholder for getting files in the PR
    # get the files in the data folder with path
    for file in Path("./data").glob("*.py"):
        print(file)
    return [str(file) for file in Path("./data").glob("*.py")]

In [112]:
set_pr_status("BLOCKED")

'PR status set to BLOCKED'

In [89]:
functions={
    "read_file": read_file,
    "post_review_comment": post_review_comment,
    "set_pr_status": set_pr_status,
    "get_files_in_pr": get_files_in_pr
}

In [106]:
tools =[
   { 
       "type":"function",
        "function":{
            "name":"read_file",
            "description":"Reads the content of a single file given its path.",
            "parameters":{
                "type":"object",
                "properties":{
                    "file_path":{
                        "type":"string",
                        "description":"The path to the file to be read."
                    }
                },
                "required":["file_path"]
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"post_review_comment",
            "description":"Posts a review comment on a specific file in the PR.",
            "parameters":{
                "type":"object",
                "properties":{
                    "filename":{
                        "type":"string",
                        "description":"The name of the file to comment on."
                    },
                    "comment":{
                        "type":"string",
                        "description":"The content of the review comment."
                    }
                },
                "required":["filename", "comment"]
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"set_pr_status",
            "description":"Sets the status of the PR to either PASS or BLOCKED.",
            "parameters":{
                "type":"object",
                "properties":{
                    "status":{
                        "type":"string",
                        "enum":["PASS", "BLOCKED"],
                        "description":"The new status for the PR."
                    }
                },
                "required":["status"]
            }
        }
    }
]

In [ ]:
def run_agent(question, max_steps=10):
    # step 1: prepare the iniial message
    messages =[
        {   "role": "system", 
            "content": """
            You are a code review assistant. First check the ./data/team_standards.md file for our team coding standards. 
            Then review the files ./data/auth.py, ./data/order.py, and ./data/payment_utils.py, and post comments for the confirmed important issues.
            Set PR status to PASS if the code is good and BLOCKED if there are issues.
            Finaly, summarize the review in a comment and post it to the PR.
            """
        },
        {
            "role": "user",
            "content": question
        }
    ]
    step = 0
    while step < max_steps:
        step += 1
        print(f"Step {step}: Sending messages to the model...")
        # Here you would typically call the language model with the messages
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        response_message = response.choices[0].message
        messages.append(response_message)
        if not response_message.tool_calls:
            print("No tool calls in the response. Ending the loop.")
            return response_message.content

        for call in response_message.tool_calls:
            function_name = call.function.name
            print(f"Function call detected: {function_name}")
            try:
                function_args = json.loads(call.function.arguments)
                print(f"Calling function: {function_name} with arguments: {function_args}")
                if function_name in functions:
                    result = functions[function_name](**function_args)
                    messages.append({
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": str(result)
                    })
                else:
                    raise ValueError(f"Function {function_name} is not defined.")
            except json.JSONDecodeError:
                print(f"Error decoding arguments for function: {function_name}")
                continue

In [114]:
pr = {
     "number": 247,
    "comments": [],
    "status": "PENDING"
}
pr["comments"] = []
pr["status"] = "PENDING"

answer = run_agent("Review PR #247 using our team coding standards.")

Step 1: Sending messages to the model...
Function call detected: read_file
Calling function: read_file with arguments: {'file_path': './data/team_standards.md'}
Step 2: Sending messages to the model...
Function call detected: read_file
Calling function: read_file with arguments: {'file_path': './data/auth.py'}
Function call detected: read_file
Calling function: read_file with arguments: {'file_path': './data/order.py'}
Function call detected: read_file
Calling function: read_file with arguments: {'file_path': './data/payment_utils.py'}
Step 3: Sending messages to the model...
Function call detected: post_review_comment
Calling function: post_review_comment with arguments: {'filename': './data/auth.py', 'comment': 'The login function currently uses a hard-coded admin check. It should be improved by validating user input against a database and using parameterized queries to prevent SQL injection.'}
Function call detected: post_review_comment
Calling function: post_review_comment with arg

BadRequestError: Error code: 400 - Invalid value for 'content': expected a string, got null.

In [109]:
print(pr)

{'number': 247, 'comments': [{'filename': './data/auth.py', 'comment': 'Issue: The token generation is not using JWT with an expiry. Consider using a library like PyJWT to properly handle JWT creation.'}, {'filename': './data/order.py', 'comment': 'Issue: No validation of user input before executing DB queries. Ensure you validate all inputs.'}, {'filename': './data/payment_utils.py', 'comment': 'Issue 1: SQL queries are not parameterized. This opens the code to SQL injection vulnerabilities. Use parameterized queries with SQLite or SQLAlchemy.\nIssue 2: Sensitive data (card number) is being logged in the print statement. Avoid logging sensitive information.'}], 'status': 'PENDING'}
